## 1. Create a connection to the database using the sqlite3 library.


In [22]:
import pandas as pd
import sqlite3
conn = sqlite3.connect('../data/checking-logs.sqlite')

## 2. Create a new table called datamart in the database by joining the tables pageviews and checker using only one query.
The table should have the following columns: "uid", "labname", "first_commit_ts", and "first_view_ts".
"first_commit_ts" is a new name for the "timestamp" column in the checker table. It shows the first commit from a particular lab and user.
"first_view_ts" shows the first time a user visited the pageviews table. It is the timestamp of when a user visited the newsfeed.
status = 'ready' should still be a filter.
numTrials = 1 should still be a filter.
"labnames" should be from the list: laba04, laba04s, laba05, laba06, laba06s, and project1.
The table should contain only users (uids with user_*), not admins.
"first_commit_ts" and "first_view_ts" should be parsed as datetime64[ns].


In [23]:
query  = """
SELECT 
    c.uid, 
    c.labname, 
    c.timestamp AS first_commit_ts, 
    MIN(p.datetime) AS first_view_ts
FROM (
    SELECT uid, labname, timestamp
    FROM checker
    WHERE status = 'ready'
    AND numTrials = 1
    AND labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
    AND uid LIKE 'user_%'
) c
LEFT JOIN pageviews p ON c.uid = p.uid
GROUP BY c.uid, c.labname
"""

datamart = pd.io.sql.read_sql(query, conn, parse_dates=['first_view_ts', 'first_commit_ts'])

datamart.to_sql('datamart', conn, if_exists='replace', index=False)
datamart.info

<bound method DataFrame.info of         uid   labname            first_commit_ts              first_view_ts
0    user_1    laba04 2020-04-26 17:06:18.462708 2020-04-26 21:53:59.624136
1    user_1   laba04s 2020-04-26 17:12:11.843671 2020-04-26 21:53:59.624136
2    user_1    laba05 2020-05-02 19:15:18.540185 2020-04-26 21:53:59.624136
3    user_1    laba06 2020-05-17 16:26:35.268534 2020-04-26 21:53:59.624136
4    user_1   laba06s 2020-05-20 12:23:37.289724 2020-04-26 21:53:59.624136
..      ...       ...                        ...                        ...
135  user_8   laba04s 2020-04-19 10:22:35.761944                        NaT
136  user_8    laba05 2020-05-02 13:28:07.705193                        NaT
137  user_8    laba06 2020-05-16 17:56:15.755553                        NaT
138  user_8   laba06s 2020-05-16 20:01:07.900727                        NaT
139  user_8  project1 2020-05-14 15:42:04.002981                        NaT

[140 rows x 4 columns]>

In [24]:
test = datamart[datamart['first_view_ts'].notna()].copy()
control = datamart[datamart['first_view_ts'].isna()].copy()
control['first_view_ts'] = test['first_view_ts'].median()
test.to_sql('test', conn, if_exists='replace', index=False)
control.to_sql('control', conn, if_exists='replace', index=False)
conn.close()